In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service as EdgeService
import time
import pandas as pd
import random
import os
import re

# ===========================
# CẤU HÌNH
# ===========================
POST_URL = "https://www.facebook.com/share/p/17aL7UkkCz/"
TARGET_NODES = 300       # Mục tiêu số người dùng DUY NHẤT
MAX_RUNTIME_MINUTES = 20 # Chạy tối đa 20 phút

edge_driver_path = "msedgedriver.exe" 

# Biến toàn cục để lưu dữ liệu
all_nodes = {} # Dùng dict để tự động lọc trùng ID: {id: name}
all_edges = []
processed_elements = set() # Lưu các thẻ HTML đã xử lý để không quét lại

# ===========================
# HÀM HỖ TRỢ
# ===========================

def extract_id(url, name):
    if not url: return f"{name}_{random.randint(1000,9999)}"
    if "profile.php" in url:
        match = re.search(r'id=(\d+)', url)
        if match: return match.group(1)
    clean_url = url.split('?')[0].split('&')[0]
    parts = clean_url.split('facebook.com/')
    if len(parts) > 1:
        return parts[1].replace('/', '')
    return f"{name}_{random.randint(1000,9999)}"

def save_data_temp():
    """Lưu file tạm thời mỗi khi quét được"""
    if not os.path.exists('database'): os.makedirs('database')
    
    # Chuyển nodes dict thành list
    nodes_list = [{"Id": k, "Label": v} for k, v in all_nodes.items()]
    
    pd.DataFrame(nodes_list).to_csv("test_database/fb_nodes.csv", index=False)
    pd.DataFrame(all_edges).to_csv("test_database/fb_edges.csv", index=False)
    print(f"💾 Đã lưu tạm: {len(nodes_list)} nodes, {len(all_edges)} edges.")

# ===========================
# CHƯƠNG TRÌNH CHÍNH
# ===========================

try:
    service = EdgeService(executable_path=edge_driver_path)
    driver = webdriver.Edge(service=service)
    driver.maximize_window()
    
    driver.get("https://www.facebook.com")
    print(f"🔥 VUI LÒNG ĐĂNG NHẬP TRONG 60 GIÂY...")
    time.sleep(60)

    driver.get(POST_URL)
    time.sleep(5)

    # 1. Chuyển sang All comments
    print("--- Chuyển chế độ 'All comments' ---")
    try:
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(2)
        dropdowns = driver.find_elements(By.XPATH, "//div[@role='button']//span[contains(text(), 'Most relevant') or contains(text(), 'Phù hợp nhất') or contains(text(), 'Top comments')]")
        if dropdowns:
            driver.execute_script("arguments[0].click();", dropdowns[0])
            time.sleep(3)
            opts = driver.find_elements(By.XPATH, "//div[@role='menuitem']//span[contains(text(), 'All comments') or contains(text(), 'Tất cả bình luận')]")
            if opts:
                driver.execute_script("arguments[0].click();", opts[0])
                print("✅ Đã chuyển All comments.")
                time.sleep(5)
    except: pass

    # 2. VÒNG LẶP CHÍNH: CUỘN - MỞ - QUÉT
    print(f"\n--- BẮT ĐẦU CHIẾN THUẬT CUỘN & QUÉT ---")
    start_time = time.time()
    
    previous_user_id = None # Để tạo cạnh nối tiếp
    
    while len(all_nodes) < TARGET_NODES:
        # Kiểm tra thời gian
        if (time.time() - start_time) / 60 > MAX_RUNTIME_MINUTES:
            print("⏰ Hết giờ!")
            break

        # A. CLICK MỞ RỘNG (Ưu tiên nút "Xem thêm" ở dưới cùng)
        try:
            # Tìm nút "Xem thêm bình luận" hoặc "View more comments"
            more_btns = driver.find_elements(By.XPATH, "//span[contains(text(), 'View more comments') or contains(text(), 'Xem thêm bình luận')]")
            clicked = False
            for btn in more_btns:
                if btn.is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                    time.sleep(0.5)
                    driver.execute_script("arguments[0].click();", btn)
                    print("⬇️  Click 'Xem thêm'")
                    clicked = True
                    time.sleep(3) # Chờ load
                    break
            
            if not clicked:
                # Nếu không có nút xem thêm, thử mở Reply
                reply_btns = driver.find_elements(By.XPATH, "//div[@role='button']//span[contains(text(), 'repl') or contains(text(), 'trả lời')]")
                real_btns = [b for b in reply_btns if ("View" in b.text or "Xem" in b.text)]
                if real_btns:
                    # Mở ngẫu nhiên 3 reply
                    for _ in range(3):
                        if not real_btns: break
                        btn = random.choice(real_btns)
                        try:
                            driver.execute_script("arguments[0].scrollIntoView({block: 'center', inline: 'nearest'});", btn)
                            driver.execute_script("arguments[0].click();", btn)
                            time.sleep(0.5)
                        except: pass
        except: pass

        # B. THU HOẠCH DỮ LIỆU (QUÉT NGAY LẬP TỨC)
        try:
            # Tìm tất cả thẻ user hiện có trên màn hình
            current_elements = driver.find_elements(By.XPATH, "//div[@role='article']//span/a[@role='link']")
            
            new_data_found = False
            
            for elem in current_elements:
                try:
                    # Dùng ID của phần tử web làm key để không xử lý lại cái đã qua
                    elem_id = elem.id 
                    if elem_id in processed_elements: continue
                    
                    processed_elements.add(elem_id) # Đánh dấu đã xử lý
                    
                    name = elem.text
                    link = elem.get_attribute('href')
                    
                    if not name or len(name) < 2: continue
                    if "hashtag" in link: continue

                    # Xử lý ID
                    user_id = extract_id(link, name)
                    
                    # 1. Lưu Node (Tự động lọc trùng nhờ Dict)
                    if user_id not in all_nodes:
                        all_nodes[user_id] = name
                        new_data_found = True
                    
                    # 2. Lưu Edge (Nối với người trước đó)
                    if previous_user_id and previous_user_id != user_id:
                        all_edges.append({
                            "Source": user_id,
                            "Target": previous_user_id,
                            "Type": "Directed"
                        })
                    
                    previous_user_id = user_id
                    
                except: continue
            
            if new_data_found:
                print(f"📊 Tiến độ: {len(all_nodes)}/{TARGET_NODES} unique users.")
                # Lưu backup mỗi khi có thêm 10 người mới
                if len(all_nodes) % 10 == 0:
                    save_data_temp()
            else:
                # Nếu không thấy gì mới, cuộn trang mạnh hơn
                driver.execute_script("window.scrollBy(0, 500);")
                time.sleep(1)

        except: pass

    # Lưu lần cuối
    save_data_temp()
    print(f"\n🎉 HOÀN THÀNH! Tổng cộng: {len(all_nodes)} users.")

except Exception as e:
    print(f"❌ Lỗi: {e}")
    # Vẫn lưu dữ liệu đã cào được trước khi lỗi
    save_data_temp()
finally:
    driver.quit()

🔥 VUI LÒNG ĐĂNG NHẬP TRONG 60 GIÂY...

--- BẮT ĐẦU CHIẾN THUẬT CUỘN & QUÉT ---
📊 Tiến độ: 24/300 unique users.
📊 Tiến độ: 28/300 unique users.
📊 Tiến độ: 33/300 unique users.
📊 Tiến độ: 41/300 unique users.
📊 Tiến độ: 44/300 unique users.
📊 Tiến độ: 45/300 unique users.
📊 Tiến độ: 61/300 unique users.
📊 Tiến độ: 67/300 unique users.
📊 Tiến độ: 76/300 unique users.
📊 Tiến độ: 84/300 unique users.
📊 Tiến độ: 102/300 unique users.
📊 Tiến độ: 103/300 unique users.
📊 Tiến độ: 112/300 unique users.
📊 Tiến độ: 126/300 unique users.
📊 Tiến độ: 145/300 unique users.
📊 Tiến độ: 146/300 unique users.
📊 Tiến độ: 148/300 unique users.
📊 Tiến độ: 159/300 unique users.
📊 Tiến độ: 162/300 unique users.
📊 Tiến độ: 163/300 unique users.
📊 Tiến độ: 164/300 unique users.
📊 Tiến độ: 172/300 unique users.
📊 Tiến độ: 179/300 unique users.
📊 Tiến độ: 188/300 unique users.
📊 Tiến độ: 204/300 unique users.
📊 Tiến độ: 211/300 unique users.
📊 Tiến độ: 221/300 unique users.
📊 Tiến độ: 229/300 unique users.
📊 Tiến 